In [4]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import pickle

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
news_df = pd.read_csv("../data/processed/news_with_topics.csv")

print("Dataset loaded successfully.")
print("Shape:", news_df.shape)
print(news_df.head())

Dataset loaded successfully.
Shape: (44898, 8)
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date label                                          full_text  \
0  December 31, 2017  Fake   Donald Trump Sends Out Embarrassing New Year’...   
1  December 31, 2017  Fake   Drunk Bragging Trump Staffer Started Rus

In [7]:
print(news_df.columns.tolist())

['title', 'text', 'subject', 'date', 'label', 'full_text', 'clean_text', 'dominant_topic']


In [8]:
news_df["semantic_text"] = (
    news_df["title"].fillna("") +
    " " +
    news_df["text"].fillna("")
)

print(news_df["semantic_text"].head())

0     Donald Trump Sends Out Embarrassing New Year’...
1     Drunk Bragging Trump Staffer Started Russian ...
2     Sheriff David Clarke Becomes An Internet Joke...
3     Trump Is So Obsessed He Even Has Obama’s Name...
4     Pope Francis Just Called Out Donald Trump Dur...
Name: semantic_text, dtype: object


In [9]:
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer loaded successfully.


In [10]:
topic_descriptions = {
    1: "Social issues, race, gender and media",
    2: "Law, courts, immigration and government",
    3: "Middle East conflict, military and international security",
    4: "International relations, nuclear issues and trade",
    5: "Crime, security and social incidents",
    6: "US elections and political campaigns",
    7: "Clinton, FBI and political controversies",
    8: "US Congress, taxes, healthcare and Republicans",
    9: "Trump, Russia and political investigations",
    10: "Trump, Republicans and political news"
}

print("Topic descriptions created.")

Topic descriptions created.


In [11]:
topic_numbers = list(topic_descriptions.keys())
topic_texts = list(topic_descriptions.values())

topic_embeddings = semantic_model.encode(
    topic_texts,
    convert_to_numpy=True
)

print("Topic embeddings created.")
print("Embedding shape:", topic_embeddings.shape)

Topic embeddings created.
Embedding shape: (10, 384)


In [12]:
sample_text = news_df["semantic_text"].iloc[0]

sample_embedding = semantic_model.encode(
    [sample_text],
    convert_to_numpy=True
)

similarities = cosine_similarity(
    sample_embedding,
    topic_embeddings
)[0]

best_topic_index = similarities.argmax()

best_topic = topic_numbers[best_topic_index]
best_score = similarities[best_topic_index]

print("Sample article:")
print(sample_text[:500])

print("\nPredicted semantic topic:", best_topic)
print("Similarity score:", round(float(best_score), 4))
print("Topic description:", topic_descriptions[best_topic])

Sample article:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, 

Predicted semantic topic: 10
Similarity score: 0.385
Topic description: Trump, Republicans and political news


In [13]:
article_embeddings = semantic_model.encode(
    news_df["semantic_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Article embeddings created successfully.")
print("Embedding shape:", article_embeddings.shape)

Batches:   0%|          | 0/1404 [00:00<?, ?it/s]

Article embeddings created successfully.
Embedding shape: (44898, 384)


In [14]:
all_similarities = cosine_similarity(
    article_embeddings,
    topic_embeddings
)

print("Similarity calculation completed.")
print("Similarity matrix shape:", all_similarities.shape)

Similarity calculation completed.
Similarity matrix shape: (44898, 10)


In [15]:
news_df["semantic_topic"] = (
    all_similarities.argmax(axis=1) + 1
)

news_df["semantic_similarity"] = (
    all_similarities.max(axis=1).round(4)
)

print("Semantic topics assigned successfully.")
print(
    news_df[
        ["dominant_topic", "semantic_topic", "semantic_similarity"]
    ].head(10)
)

Semantic topics assigned successfully.
   dominant_topic  semantic_topic  semantic_similarity
0              10              10               0.3850
1               9               9               0.4995
2               1               7               0.3280
3              10               9               0.2965
4               1               3               0.2602
5               1               5               0.2575
6               9               7               0.5514
7              10               7               0.3637
8              10               9               0.5462
9               1              10               0.3304


In [16]:
agreement = (
    news_df["dominant_topic"] ==
    news_df["semantic_topic"]
)

agreement_percentage = agreement.mean() * 100

print(
    f"LDA vs Semantic Topic Agreement: "
    f"{agreement_percentage:.2f}%"
)

LDA vs Semantic Topic Agreement: 38.61%


In [17]:
agreement = (
    news_df["dominant_topic"] ==
    news_df["semantic_topic"]
)

agreement_percentage = agreement.mean() * 100

print(
    f"LDA vs Semantic Topic Agreement: "
    f"{agreement_percentage:.2f}%"
)

LDA vs Semantic Topic Agreement: 38.61%


In [18]:
comparison_table = pd.crosstab(
    news_df["dominant_topic"],
    news_df["semantic_topic"],
    normalize="index"
) * 100

comparison_table = comparison_table.round(2)

print("LDA vs Semantic Topic Distribution (%)")
print(comparison_table)

LDA vs Semantic Topic Distribution (%)
semantic_topic     1      2      3      4      5      6      7      8      9   \
dominant_topic                                                                  
1               22.07   3.17   3.72   0.60  10.49   7.95  31.82   2.99   7.55   
2                2.39  31.29   7.12  14.27   2.09  10.65  13.75   5.35   9.42   
3                0.16   1.66  73.61   5.43   2.18   1.70   3.01   0.63   9.75   
4                0.43   1.50  22.20  56.77   0.39   1.16   2.56   1.52  10.86   
5                3.77   9.29  29.27   3.29  22.01   4.64  10.86   0.37  13.82   
6                2.51   2.83   0.96   7.93   0.32  39.87  18.17   5.77  10.02   
7                1.06   0.70   1.79   0.83   1.52   8.58  51.49   0.46  21.89   
8                1.41   2.90   1.34   2.15   3.28   5.11  11.23  61.35   6.29   
9                0.14   0.61   1.39   1.45   0.22   1.78  34.72   1.48  54.26   
10               1.28   0.72   1.22   1.02   0.01  18.33  33.00   4.14

In [19]:
semantic_stats = pd.Series({
    "Mean Similarity": all_similarities.max(axis=1).mean(),
    "Median Similarity": np.median(all_similarities.max(axis=1)),
    "Minimum Similarity": all_similarities.max(axis=1).min(),
    "Maximum Similarity": all_similarities.max(axis=1).max()
})

print(semantic_stats.round(4))

Mean Similarity       0.3237
Median Similarity     0.3176
Minimum Similarity    0.0056
Maximum Similarity    0.7103
dtype: float32


In [20]:
import pickle

semantic_artifacts = {
    "topic_descriptions": topic_descriptions,
    "topic_embeddings": topic_embeddings
}

with open("../models/semantic_topic_embeddings.pkl", "wb") as f:
    pickle.dump(semantic_artifacts, f)

print("Semantic topic artifacts saved successfully.")

Semantic topic artifacts saved successfully.
